In [ ]:
import operator
from langgraph.graph import END, START, StateGraph
from typing import TypedDict
import subprocess
from openai import OpenAI
from langchain.chat_models import init_chat_model
from typing_extensions import Annotated
import operator
import base64

import dotenv
dotenv.load_dotenv()

llm = init_chat_model("openai:gpt-4o-mini")

class State(TypedDict):
    video_file: str
    audio_file: str
    transcription : str
    summaries : Annotated[list[str], operator.add]

In [ ]:
def extract_audio(state: State):
    output_file = state["video_file"].replace("mp4", "mp3")
    command = [
        "ffmpeg",
        "-y",  # 확인 없이 기존 파일 덮어쓰기
        "-i",
        state["video_file"],
        "-filter:a",
        "atempo=2.0",
        output_file
    ]
    subprocess.run(command)
    return {
        "audio_file" : output_file,
    }

In [ ]:
def transcribe_audio(state: State):
    client = OpenAI()
    with open(state["audio_file"], "rb") as audio_file:
        transcription = client.audio.transcriptions.create(
            model="whisper-1",
            response_format="text",
            file=audio_file,
            language="en",
            prompt="jennie", 
        )
        return {
            'transcription':transcription,
        }

In [ ]:
import textwrap
from langgraph.types import Send

def dispatch_summarizers(state: State):
    transcription = state['transcription']
    chunks = []
    for i, chunk in enumerate(textwrap.wrap(transcription, 500)):
        chunks.append({"id": i+1, "chunk" : chunk})
    return [Send("summarize_chunk", chunk) for chunk in chunks]

In [ ]:

def summarize_chunk(chunk):
    chunk_id = chunk["id"]
    chunk = chunk["chunk"]

    response = llm.invoke(
        f"""
        Please summarize the following text.

        Text: {chunk}
        """
    )
    summary = f"[Chunk {chunk_id}] {response.content}"
    return {
        "summaries" : [summary],
    }

In [ ]:
graph_builder = StateGraph(State)

graph_builder.add_node("extract_audio", extract_audio)
graph_builder.add_node("transcribe_audio", transcribe_audio)
graph_builder.add_node("summarize_chunk", summarize_chunk)

graph_builder.add_edge(START, "extract_audio")
graph_builder.add_edge("extract_audio", "transcribe_audio")
graph_builder.add_conditional_edges("transcribe_audio", dispatch_summarizers, ["summarize_chunk"])
graph_builder.add_edge("summarize_chunk", END)

graph = graph_builder.compile()

In [ ]:
graph.invoke({"video_file": "video.mp4"})